# Imports

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
import numpy as np
import subprocess

warnings.filterwarnings('ignore')


# Data Cleaning

In [5]:
drop_cols = [
    'device_name', 'device_mac', 'label_full',
    'label1', 'label2', 'label3', 'label4',
    'timestamp', 'timestamp_start', 'timestamp_end',
    'log_data-types', 'log_interval-messages',
    'network_ips_all', 'network_ips_dst', 'network_ips_src',
    'network_macs_all', 'network_macs_dst', 'network_macs_src',
    'network_ports_all', 'network_ports_dst', 'network_ports_src',
    'network_protocols_all', 'network_protocols_dst', 'network_protocols_src',
]

dataFile_b = pd.read_csv('benignDataCombinedFile.csv')
dataFile_b = dataFile_b.drop(columns=drop_cols, errors='ignore')
dataFile_b['label'] = 'benign'

attack_data = []
for i in range(1,11):
    dataFile_a = pd.read_csv(f'Attack Data/attack_samples_{i}sec.csv')
    dataFile_a = dataFile_a.drop(columns=drop_cols, errors='ignore')
    dataFile_a['label'] = 'attack'
    attack_data.append(dataFile_a)

data = pd.concat([dataFile_b] + attack_data, ignore_index=True)

In [8]:
## Training
print(data.head(10))
X = data.drop(columns=['label'])

y = data['label'].map({'benign': 0, 'attack': 1})

X = X.fillna(X.median())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train, y_train)



   log_data-ranges_avg  log_data-ranges_max  log_data-ranges_min  \
0                  0.0                  0.0                  0.0   
1                  0.0                  0.0                  0.0   
2                  0.0                  0.0                  0.0   
3                  0.0                  0.0                  0.0   
4                  0.0                  0.0                  0.0   
5                  0.0                  0.0                  0.0   
6                  0.0                  0.0                  0.0   
7                  0.0                  0.0                  0.0   
8                  0.0                  0.0                  0.0   
9                  0.0                  0.0                  0.0   

   log_data-ranges_std_deviation  log_data-types_count  log_messages_count  \
0                            0.0                     0                   0   
1                            0.0                     0                   0   
2                

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [10]:
## Prediction
y_pred = rf_classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred, target_names=['Benign', 'Attack'])

print(f"Accuracy: {accuracy:.2f}")
print("\nClassification Report:\n", classification_rep)

Accuracy: 0.97

Classification Report:
               precision    recall  f1-score   support

      Benign       0.96      1.00      0.98     79901
      Attack       1.00      0.94      0.97     57234

    accuracy                           0.97    137135
   macro avg       0.98      0.97      0.97    137135
weighted avg       0.97      0.97      0.97    137135



In [13]:
## Sample Testing Displayed
sampleSize = 100

for i in range(sampleSize):
    sample = X_test.iloc[i:i+1]
    actual = y_test.iloc[i]
    prediction = rf_classifier.predict(sample)

    if (prediction == actual):
        continue

    sample_dict = sample.iloc[0].to_dict()
    print(f"\nSample Passenger: {sample_dict}")
    print(f"Predicted: {'Attack' if prediction[0] == 1 else 'Benign'}")
    print(f"Actual: {'Attack' if actual == 1 else 'Benign'}")


Sample Passenger: {'log_data-ranges_avg': 0.0, 'log_data-ranges_max': 0.0, 'log_data-ranges_min': 0.0, 'log_data-ranges_std_deviation': 0.0, 'log_data-types_count': 0.0, 'log_messages_count': 0.0, 'network_fragmentation-score': 0.0, 'network_fragmented-packets': 0.0, 'network_header-length_avg': 0.0, 'network_header-length_max': 0.0, 'network_header-length_min': 0.0, 'network_header-length_std_deviation': 0.0, 'network_interval-packets': 0.0, 'network_ip-flags_avg': 0.0, 'network_ip-flags_max': 0.0, 'network_ip-flags_min': 0.0, 'network_ip-flags_std_deviation': 0.0, 'network_ip-length_avg': 0.0, 'network_ip-length_max': 0.0, 'network_ip-length_min': 0.0, 'network_ip-length_std_deviation': 0.0, 'network_ips_all_count': 0.0, 'network_ips_dst_count': 0.0, 'network_ips_src_count': 0.0, 'network_macs_all_count': 0.0, 'network_macs_dst_count': 0.0, 'network_macs_src_count': 0.0, 'network_mss_avg': 0.0, 'network_mss_max': 0.0, 'network_mss_min': 0.0, 'network_mss_std_deviation': 0.0, 'networ

In [17]:
TSHARK_PATH = r'C:\Program Files\Wireshark\tshark.exe'

def capture_packets(interface='Wi-Fi', duration=10):
    print("[i] Capturing traffic...")
    cmd = [
        TSHARK_PATH, '-i', interface, '-T', 'fields',
        '-e', 'frame.time_epoch',
        '-e', 'ip.ttl',
        '-e', 'ip.hdr_len',
        '-e', 'ip.len',
        '-e', 'ip.flags.mf',
        '-e', 'ip.frag_offset',
        '-E', 'header=y', '-E', 'separator=,',
        '-a', f'duration:{duration}'
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    from io import StringIO
    packets = pd.read_csv(StringIO(result.stdout))
    return packets

In [28]:
def aggregate_features(packets):
    features = {}
    
    # Example aggregations — adapt these to match YOUR actual columns
    features['network_ttl_avg'] = packets['ip.ttl'].mean()
    features['network_ttl_max'] = packets['ip.ttl'].max()
    features['network_ttl_min'] = packets['ip.ttl'].min()
    features['network_header_length_avg'] = packets['ip.hdr_len'].mean()
    features['network_header_length_max'] = packets['ip.hdr_len'].max()
    features['network_fragmented_packets'] = (packets['ip.frag_offset'] > 0).sum()
    features['network_fragmentation_score'] = features['network_fragmented_packets'] / len(packets)
    features['log_messages_count'] = len(packets)
    
    # Fill any features your model expects but you can't derive with 0
    return features

def predict_traffic(rf_classifier, feature_columns, interface='eth0', duration=10):
    packets = capture_packets(interface, duration)
    
    if packets.empty:
        print("No packets captured.")
        return
    
    features = aggregate_features(packets)
    
    # Build a DataFrame with the exact columns the model expects
    sample = pd.DataFrame([features])
    
    # Add missing columns as 0, reorder to match training data
    for col in feature_columns:
        if col not in sample.columns:
            sample[col] = 0.0
    sample = sample[feature_columns]
    print("[i] Predicting...")
    prediction = rf_classifier.predict(sample)
    print(f"Captured {len(packets)} packets over {duration}s")
    print(f"Prediction: {'Attack' if prediction[0] == 1 else 'Benign'}")

# Use it — pass in your trained model and the feature column list
feature_columns = X_train.columns.tolist()
predict_traffic(rf_classifier, feature_columns, interface='Wi-Fi', duration=10)

[i] Capturing traffic...
[i] Predicting...
Captured 159 packets over 10s
Prediction: Benign


In [22]:
import subprocess

result = subprocess.run(
    [r'C:\Program Files\Wireshark\tshark.exe', '-D'],
    capture_output=True, text=True
)
print(result.stdout)

1. \Device\NPF_{1A3AAB2D-E408-45FD-A353-359D3C26058D} (Local Area Connection* 10)
2. \Device\NPF_{00EAD091-DFEE-47B7-B1E0-69454EB2D8E0} (Local Area Connection* 9)
3. \Device\NPF_{D19CD7B1-BC6A-42A4-88D7-3BE5BAAD825B} (Local Area Connection* 8)
4. \Device\NPF_{65A88340-B130-4F0B-AE5B-25C84AC22255} (Bluetooth Network Connection)
5. \Device\NPF_{2F717AEC-37C9-41CD-A01F-CECA7025AB57} (Wi-Fi)
6. \Device\NPF_{7E5CFAF7-BC02-49E8-93EF-313D08B5631B} (Local Area Connection* 2)
7. \Device\NPF_{12BF4050-537D-444B-AF85-D9F46B0750E6} (Local Area Connection* 1)
8. \Device\NPF_{4334376A-4658-48C0-BBC8-8979DD41D584} (Ethernet 4)
9. \Device\NPF_{9C752E0F-6EEF-4C30-A973-1DF5950A5A96} (vEthernet (WSL (Hyper-V firewall)))
10. \Device\NPF_{B9DAE269-3AF5-40F7-8460-A2625F005311} (Local Area Connection)
11. \Device\NPF_Loopback (Adapter for loopback traffic capture)
12. \Device\NPF_{877F2D1C-2CE6-4653-99C8-9EB6943FE4DA} (OpenVPN Connect DCO Adapter)
13. \\.\USBPcap1 (USBPcap1)
14. \\.\USBPcap2 (USBPcap2)
15. \